<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/GEM_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
# @title Dependencies

# first-party
from typing import Dict, Any, Optional, Tuple, List
import random
import time
import csv
import os

# third-party
from google.colab import drive
import pandas as pd
from openai import OpenAI


In [ ]:
# @title Paths and definitions

# mount GDrive
drive.mount('/content/drive')

# dataset directory
DATASET_DIR = "/content/drive/MyDrive/Thesis/dataset/"
# create dataset directory (if not already existing)
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# raw directory
RAW_DIR = "/content/drive/MyDrive/Thesis/raw/"
# create raw directory (if not already existing)
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# define result directory
RESULTS_DIR = RAW_DIR # Using RAW_DIR as RESULTS_DIR is the same path

# define path directory
LOG_DIR = RAW_DIR # Using RAW_DIR as LOG_DIR is the same path

# system prompt for judgement processing
SYSTEM_PROMPT_PROCESSING = """

Carefully read the text of a scientific paper review. You should summarize each evaluation in the review in a separate line.
Begin each summary line with one of the following phrases: ’The reviewer appreciates’, ’The reviewer criticizes’, ’The reviewer questions’, ’The reviewer suggests’.
You need to keep the summary as concise as possible, excluding specific details about the paper’s content, such as topics, ideas, methods, findings, and any mathematical symbols.
You should ensure that even if multiple evaluations are mentioned in the same sentence in the original review, you should still split it into separate lines.
For example, you should not output a line like ’The reviewer appreciates the well-written paper and good experimental performance’.
In contrast, you should output ’The reviewer appreciates the well-written paper’ and ’The reviewer appreciates good experimental performance’ in two lines.

"""

OPENAI_KEY = "Set key here"

SIMULATE = True  # Set to False to make real API calls

MAX_RETRIES = 3

RETRY_DELAY = 2.0

PREPROCESSING_MODEL = "gpt-4o-2024-05-13"

# human reviews
DATASET_PAPER = "dataset_paper.parquet"

# llm reviews
LLM_REVIEW = 'llm_sim_results.csv'

# define human review columns
HUMAN_REVIEW_COLUMNS = ['human_review1', 'human_review2', 'human_review3']

# define llm review column
LLM_REVIEW_COLUMN = 'review_text'

# define columns of result file
BASE_STAND_REVW_COLMNS = ['paper_id',
                          'standardised_review_text',
                          "standardisation_status",
                          "error_message",
                          "processed_timestamp",
                          'model', 'temperature', 'reasoning_effort', 'chain_of_thought']

# define columns of log file
BASE_LOG_COLUMNS = ['model',
                    'temperature',
                    'reasoning_effort',
                    'chain_of_thought',
                    'papers_processed_count',
                    'papers_succeeded_count',
                    'configuration_status']

# result file
STAND_REVW_CSV = "standardise_human_review_1_res.csv"

# log file
LOG_FILE_CSV = "standardise_human_review_1_log.csv"


In [3]:
# @title Load ICLR data

# full path
FULL_PATH_ICLR = DATASET_DIR + DATASET_PAPER

try:
    # load the Parquet file into a Pandas DataFrame
    df_reviews_source = pd.read_parquet(FULL_PATH_ICLR)

    # check
    print("\nDataFrame Head:")
    print(df_reviews_source.head())

    print("\nDataFrame Columns:")
    print(df_reviews_source.columns.tolist())

# error message for file not found
except FileNotFoundError:
    print(f"ERROR: Parquet file not found at {FULL_PATH_ICLR}. Please check the path.")

# error message for file not readable
except Exception as e:
    print(f"An error occurred while reading the Parquet file: {e}")


DataFrame Columns:
['paper_id', 'pdf_url', 'abstract', 'parsed_text', 'human_review1', 'human_review2', 'human_review3']


In [ ]:
# @title Load LLM reviews

# full path
FULL_PATH_LLM = RAW_DIR + LLM_REVIEW

# enforce corrrect data types of directory
LLM_CSV_DTYPES = {
    'document_id': str,
    'review_text': str,
    'raw_api_output': str,
    'model': str,
    'temperature': float,
    'reasoning_effort': str,
    'chain_of_thought': str,
    'CoT': str
}

try:
    # load the CSV file into a Pandas DataFrame
    df_llm_results = pd.read_csv(FULL_PATH_LLM, dtype=LLM_CSV_DTYPES)

    # check
    print("\nDataFrame Head:")
    print(df_llm_results.head())

    print("\nDataFrame Columns:")
    print(df_llm_results.columns)

# error message for file not found
except FileNotFoundError:
    print(f"ERROR: Parquet file not found at {FULL_PATH_LLM}. Please check the path.")

# error message for file not readable
except Exception as e:
    print(f"An error occurred while reading the Parquet file: {e}")


# Preprocessing

In [6]:
# @title Define simulation

# define LLMClient
class LLMClient:
    """
    Handles API calls, simulation, and retry logic (adopted from supervisor's pattern).
    """
    def __init__(self, simulate: bool = True, api_key: str = OPENAI_KEY, seed: int = 7):
        self.simulate = simulate
        self.rng = random.Random(seed)
        self._openai = None
        self.api_key = api_key

        # initialize client if not simulating
        self._maybe_init_clients()

    def _maybe_init_clients(self):
        """Initializes the real OpenAI client if not in simulation mode."""
        if self.simulate:
            print("INFO: LLMClient initialized in SIMULATION mode.")
            return

        if self.api_key == "placeholder for API-key":
             print("If simulation=FALSE set true API-key")

        try:
            # only OpenAI-API needed
            self._openai = OpenAI(api_key=self.api_key)
            print("INFO: LLMClient initialized for API calls.")
        except Exception as e:
            print(f"ERROR: Could not initialize OpenAI client: {e}")
            # fallback to simulation if initialization fails
            self.simulate = True

    def call(
        self,
        system_prompt: str,
        user_prompt: str,
        temperature: float,
        max_tokens: int,
        max_retries: int = MAX_RETRIES,
        retry_delay: float = RETRY_DELAY,
        model: str = PREPROCESSING_MODEL
        ) -> Tuple[str, str]:
        """
        Executes the API call with retry logic (adopted from supervisor's pattern).

        Returns: (raw_text, status_code/error_message)
        """

        # simulation path
        if self.simulate:
            simulated_text = (
                f"Simulated Standardization: The core argument was '{user_prompt[:50]}...'. "
                f"Model={model}, Temp={temperature}. Output is a concise summary."
            )
            return simulated_text, "SIMULATED"

        # real API call
        for attempt in range(1, max_retries + 1):
            try:
                # direct implementation of the function logic
                response = self._openai.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=temperature,
                    max_tokens=max_tokens
                )

                # success: Extract the text and return
                generated_text = response.choices[0].message.content
                return generated_text, "SUCCESS"

            except Exception as e:
                error_msg = f"API Error (Attempt {attempt}/{max_retries}): {type(e).__name__}: {e}"

                if attempt == max_retries:
                    print(f"FATAL: {error_msg}")
                    return f"ERROR: Failed after {max_retries} attempts. {e}", "FAILURE"

                print(f"WARNING: {error_msg}. Retrying in {retry_delay}s...")
                time.sleep(retry_delay)

        return "ERROR: Loop finished without success.", "FAILURE" # should not be reached


# define review call
def standardise_review(
    client: LLMClient,
    system_prompt_processing: str,
    user_prompt_processing: str,
    id_variable: str,
    original_review_model: Optional[str],
    original_review_temperature: Optional[float],
    original_reasoning_effort: Optional[str] = None,
    original_chain_of_thought: Optional[str] = None,
    max_out_tokens: int = 4000
) -> List[Dict[str, Any]]:

    # use the LLMClient's call method with hardcoded standardization model and temperature
    generated_text, status = client.call(
        model=PREPROCESSING_MODEL, # Hardcoded standardization model
        system_prompt=system_prompt_processing,
        user_prompt=user_prompt_processing,
        temperature=0.0, # Hardcoded standardization temperature
        max_tokens=max_out_tokens,
    )

    # initialize list
    records = []

    # append results to the list, including the original review's metadata
    records.append({
        'paper_id': id_variable,
        'standardised_review_text': generated_text,
        'standardisation_status': status,
        'error_message': None,
        'processed_timestamp': time.time(), # Capture timestamp of processing
        'model': original_review_model, # Store the original review's model/placeholder
        'temperature': original_review_temperature, # Store the original review's temperature/placeholder
        'reasoning_effort': original_reasoning_effort,
        'chain_of_thought': original_chain_of_thought
    })

    return records

# Helper function to process a given review column with specific config parameters
def process_review_source(
    df: pd.DataFrame,
    review_col_name: str,
    config_model: Optional[str],
    config_temperature: Optional[float],
    config_reasoning_effort: Optional[str],
    config_chain_of_thought: Optional[str],
    results_path: str,
    log_path: str
):
    print(f"\n--- Processing {review_col_name} (Model: {config_model}, Temp: {config_temperature}) ---")

    papers_processed_count = 0
    papers_succeeded_count = 0

    # Iterate over each paper in the provided dataframe/group
    for i, row in df.iterrows():
        papers_processed_count += 1

        review_text_content = str(row[review_col_name])

        try:
            df_result = standardise_review(
                client=client,
                system_prompt_processing=SYSTEM_PROMPT_PROCESSING,
                user_prompt_processing=review_text_content,
                id_variable=row['paper_id'] if 'paper_id' in row else row['document_id'],
                original_review_model=config_model, # Pass the config_model for logging
                original_review_temperature=config_temperature, # Pass the config_temperature for logging
                original_reasoning_effort=config_reasoning_effort, # Pass for logging
                original_chain_of_thought=config_chain_of_thought # Pass for logging
            )

            df_result = pd.DataFrame(df_result)
            df_result = df_result[BASE_STAND_REVW_COLMNS]
            df_result.to_csv(results_path, mode="a", header=False, index=False)

            print(f"[SUCCESS] Processed {review_col_name} for ID {row['paper_id'] if 'paper_id' in row else row['document_id']} with original config (Model: {config_model}, Temp: {config_temperature}).")
            papers_succeeded_count += 1

        except Exception as e:
            failure_record = {
                'paper_id': row['paper_id'] if 'paper_id' in row else row['document_id'],
                'standardised_review_text': None,
                'standardisation_status': "FAILURE",
                'error_message': f"Failed to process. Error: {type(e).__name__}: {str(e)}",
                'processed_timestamp': time.time(),
                'model': config_model, # Store the original review's model/placeholder
                'temperature': config_temperature, # Store the original review's temperature/placeholder
                'reasoning_effort': config_reasoning_effort,
                'chain_of_thought': config_chain_of_thought
            }
            df_result = pd.DataFrame([failure_record])
            df_result = df_result[BASE_STAND_REVW_COLMNS]
            df_result.to_csv(results_path, mode="a", header=False, index=False)
            print(f"[FAILURE] Failed to process {review_col_name} for ID {row['paper_id'] if 'paper_id' in row else row['document_id']} with original config (Model: {config_model}, Temp: {config_temperature}). Error: {e}")

    log_entry = {
        'model': config_model,
        'temperature': config_temperature,
        'reasoning_effort': config_reasoning_effort,
        'chain_of_thought': config_chain_of_thought,
        'papers_processed_count': papers_processed_count,
        'papers_succeeded_count': papers_succeeded_count,
        'configuration_status': "SUCCESS" if papers_succeeded_count == papers_processed_count else "PARTIAL_FAILURE" if papers_succeeded_count > 0 else "TOTAL_FAILURE"
    }

    # Only write log entry if processing is complete for this configuration
    log_df = pd.DataFrame([log_entry])
    log_df[BASE_LOG_COLUMNS].to_csv(log_path, mode="a", header=False, index=False)
    print(f"Logged status for {review_col_name} (Model: {config_model}, Temp: {config_temperature}): {log_entry['configuration_status']} (Processed {papers_succeeded_count}/{papers_processed_count} papers).")


In [ ]:
# @title Human review standardisation

# initialize the client
client = LLMClient(simulate=SIMULATE)

for i, col_name in enumerate(HUMAN_REVIEW_COLUMNS):
    review_number = col_name.replace('human_review', '')
    current_results_path = os.path.join(RESULTS_DIR, f"std_human_revw_{review_number}_res.csv")
    current_log_path = os.path.join(LOG_DIR, f"std_human_revw_{review_number}_log.csv")

    # Ensure result file header exists before processing starts for this human review column
    if not os.path.exists(current_results_path):
        with open(current_results_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(BASE_STAND_REVW_COLMNS)

    # Ensure log file header exists before processing starts for this human review column
    if not os.path.exists(current_log_path):
        with open(current_log_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(BASE_LOG_COLUMNS)

    # Group df_reviews_source by paper_id to process each paper individually
    grouped_human_papers = df_reviews_source.groupby('paper_id')

    for paper_id, paper_group_df in grouped_human_papers:
        process_review_source(
            df=paper_group_df, # Pass the single-row DataFrame (group of one paper)
            review_col_name=col_name,
            config_model="human", # Indicate it's a human review
            config_temperature=None, # No temperature for human reviews
            config_reasoning_effort=None, # No reasoning effort for human reviews
            config_chain_of_thought=None, # No chain of thought for human reviews
            results_path=current_results_path,
            log_path=current_log_path
        )


In [ ]:
# @title Run LLM review standardisation

# initialize the client
client = LLMClient(simulate=SIMULATE)

# Prepare df_llm_results for grouping
df_llm_results_for_grouping = df_llm_results.copy()
df_llm_results_for_grouping['temperature_group_key'] = df_llm_results_for_grouping['temperature'].fillna('NaN_temp').astype(str)
df_llm_results_for_grouping['reasoning_effort_group_key'] = df_llm_results_for_grouping['reasoning_effort'].fillna('NaN_effort').astype(str)
df_llm_results_for_grouping['chain_of_thought_group_key'] = df_llm_results_for_grouping['chain_of_thought'].fillna('NaN_CoT').astype(str)

# Group by the unique configuration parameters for LLM reviews
config_groups_llm = df_llm_results_for_grouping.groupby(['model', 'temperature_group_key', 'reasoning_effort_group_key', 'chain_of_thought_group_key'])

# Define paths for LLM review results and logs
LLM_RESULTS_PATH = os.path.join(RESULTS_DIR, "standardise_llm_review_res.csv")
LLM_LOG_PATH = os.path.join(LOG_DIR, "standardise_llm_review_log.csv")

# Ensure result file header exists before processing starts for LLM reviews
if not os.path.exists(LLM_RESULTS_PATH):
    with open(LLM_RESULTS_PATH, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(BASE_STAND_REVW_COLMNS)

# Ensure log file header exists before processing starts for LLM reviews
if not os.path.exists(LLM_LOG_PATH):
    with open(LLM_LOG_PATH, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(BASE_LOG_COLUMNS)

for config_key, config_group_df in config_groups_llm:
    current_model_key, current_temperature_key, current_reasoning_effort_key, current_chain_of_thought_key = config_key

    # Extract original values for logging from a sample row in the group
    sample_row = config_group_df.iloc[0]
    original_model_for_log = sample_row['model']
    original_temperature_for_log = sample_row['temperature']
    original_reasoning_effort_for_log = sample_row['reasoning_effort']
    original_chain_of_thought_for_log = sample_row['chain_of_thought']

    process_review_source(
        df=config_group_df, # Use the grouped LLM dataframe
        review_col_name=LLM_REVIEW_COLUMN,
        config_model=original_model_for_log,
        config_temperature=original_temperature_for_log,
        config_reasoning_effort=original_reasoning_effort_for_log,
        config_chain_of_thought=original_chain_of_thought_for_log,
        results_path=LLM_RESULTS_PATH,
        log_path=LLM_LOG_PATH
    )

## Summary of author stated strengths and weaknesses

**System prompt**

You are given a paper submission for a top-tier Machine Learning conference. Your goal is to identify and list the strengths and weaknesses that the paper claims about itself. This task requires careful reading of the paper. Please follow these steps to complete the task: 1. Carefully read the entire paper submission. As you read, identify instances where the authors mention strengths or positive aspects of their research, methodology, results, or contributions. These are the strengths claimed by the paper. Also, identify instances where the authors mention limitations, weaknesses, or areas for future improvement in their work. These are the weaknesses claimed by the paper. 2. Compile your findings into two separate lists: one for strengths and one for weaknesses. 3. For each list, write each point on a separate line, keeping it concise. Add an extra blank line between each point for clarity. 4. Format your output as follows: ¡strengths claimed by the paper¿ [List each strength claimed by the paper in separate lines, with an extra blank line between each point] ¡/strengths claimed by the paper¿ ¡weaknesses claimed by the paper¿ [List each weakness claimed by the paper in separate lines, with an extra blank line between each point] ¡/weaknesses claimed by the paper¿ Important: Focus only on the strengths and weaknesses that the paper claims about itself. Do not include your own evaluation or opinion of the paper’s merits or shortcomings. Do not include the strengths and weaknesses of the baseline. Your task is to report what the authors themselves have stated about their work’s strengths and limitations.

**User prompt**

{Full paper in Text Format}

## Judgement prediction

**System Prompt**

You are the second reviewer for a scientific paper. You are given the abstract of the paper and a list of review judgments from the first reviewer, starting with ’The reviewer appreciates/criticizes/questions/suggests’. Your task is to provide your own judgments of the paper based on the given materials. You should create a separate line for each judgment you have, starting with ’The reviewer appreciates/criticizes/questions/suggests’. Ensure your judgments are concise, excluding specific details about the paper’s content.

**User Prompt**

[Abstract of the paper]

{Abstract of the Paper if the metric is GEM-S, and “Not Available” if the metric is GEM}

[Review judgments from the first reviewer]

{Preprocessed Review Judgments (x) for conditional probability, and “Not Available” for marginal probability}

**Forced LLM Output**

{Preprocessed Review Judgments (y)}